 # 01. Cleaning rules, definitions, and the cleaning log

**What this notebook produces**
The cleaned dataset will be saved as data/clean_transactions_v2.pkl. 
A separate cleaning log will also be created in docs/cleaning-log.md.

## What one row is
Each row represents one invoice line. 

For this project, an order is defined as one customer making a purchase on a particular day. 

Multiple invoices from the same customer on the same day are therefore counted as one order.

## Cleaning rules
The following steps are used to clean the dataset:

- Missing customer ID: Rows without a customer ID are removed because they cannot be linked to a specific customer.

- Returns and cancellations: Rows where the invoice number starts with C are removed.

- Invalid transactions: Rows with a price or quantity of zero or less are removed.

- Duplicate records: Exact duplicates are removed by comparing all 8 original columns. The first occurrence is kept.

- Non-product entries: Rows with stock codes linked to fees, adjustments or test entries are removed. 
These include POST, C2, DOT, BANK CHARGES, ADJUST, ADJUST2, TEST001, TEST002 and D.

**Kept with a note (not removed):**
M (Manual) entries.
PADS and SP1002, which are treated as product codes.
Customers with unusually high spending.

## Setup and load

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option('display.max_columns', None, 'display.width', 200)

# Switches: change one and rerun the whole notebook to see what it does to the counts.
DROP_DUPLICATES = True       # R4
DROP_NONPRODUCT = True       # R5
NONPRODUCT_CODES = ['POST', 'C2', 'DOT', 'BANK CHARGES', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002', 'D']

In [2]:
xl = pd.ExcelFile('data/online_retail_II.xlsx')
raw = pd.concat([pd.read_excel(xl, sheet_name=s) for s in xl.sheet_names], ignore_index=True)
raw['Invoice'] = raw['Invoice'].astype(str)
raw['StockCode'] = raw['StockCode'].astype(str)
RAW_COLS = list(raw.columns)
print(raw.shape)                               
print(raw['InvoiceDate'].min(), '->', raw['InvoiceDate'].max())

(1067371, 8)
2009-12-01 07:45:00 -> 2011-12-09 12:50:00


## Applying the cleaning rules
To keep track of the changes, I created a function `apply_rule` that applies each rule and records its results.

For every step, the log includes:

- Number of rows removed and remaining.

- Number of customers remaining.

- Customers who no longer have any transactions.

- Customers whose first recorded purchase date changes.

- Total value of the removed transactions.

This makes it easier to see how each cleaning decision affects the dataset.

In [3]:
log = []

def apply_rule(df, remove_mask, rule, what, how, action):
    removed = df[remove_mask]
    kept = df[~remove_mask]
    first_before = df.groupby('Customer ID')['InvoiceDate'].min()
    first_after = kept.groupby('Customer ID')['InvoiceDate'].min()
    common = first_after.index
    log.append({
        'rule': rule, 'what': what, 'how_found': how, 'action': action,
        'rows_removed': len(removed), 'rows_remaining': len(kept),
        'customers_remaining': kept['Customer ID'].nunique(),
        'customers_lost': first_before.index.difference(first_after.index).size,
        'first_order_moved': int((first_after != first_before.reindex(common)).sum()),
        'value_removed': round(float((removed['Quantity'] * removed['Price']).sum()), 0),
    })
    return kept

# Add the original dataset as the starting point in the log.
log.append({'rule': 'Raw', 'what': 'file as received', 'how_found': '-', 'action': '-',
            'rows_removed': 0, 'rows_remaining': len(raw), 'customers_remaining': raw['Customer ID'].nunique(),
            'customers_lost': 0, 'first_order_moved': 0, 'value_removed': 0.0})

## Apply the rules

In [4]:
# Worked on a copy so the original dataset stays unchanged.
df = raw.copy()

# Remove transactions without a customer ID.
df = apply_rule(df, df['Customer ID'].isna(), 'R1', 'No customer ID', 'Customer ID blank', 'removed')
# Remove returns and cancellations.
df = apply_rule(df, df['Invoice'].str.startswith('C'), 'R2', 'Returns and cancellations', 'Invoice starts with C', 'removed')
# Remove rows with invalid prices or quantities.
df = apply_rule(df, (df['Price'] <= 0) | (df['Quantity'] <= 0), 'R3', 'Non-positive price or quantity',
                'Price <= 0 or Quantity <= 0', 'removed')
# Remove exact duplicates, keeping the first occurrence.
if DROP_DUPLICATES:
    df = apply_rule(df, df.duplicated(subset=RAW_COLS, keep='first'), 'R4', 'Exact duplicate lines',
                    'all 8 columns identical to an earlier line', 'later copy removed')
# Remove fees, adjustments and test entries.
if DROP_NONPRODUCT:
    df = apply_rule(df, df['StockCode'].isin(NONPRODUCT_CODES), 'R5', 'Fee, service, adjustment and test lines',
                    'StockCode in ' + ', '.join(NONPRODUCT_CODES), 'removed')

# Prepare the final dataset for analysis.
df['Customer ID'] = df['Customer ID'].astype(int)
df['TotalPrice'] = df['Quantity'] * df['Price']
# an order = one customer on one day
df['OrderDay'] = df['InvoiceDate'].dt.normalize()       
print(df.shape)

(777277, 10)


In [5]:
cleaning_log = pd.DataFrame(log)
cleaning_log[['rule', 'what', 'rows_removed', 'rows_remaining', 'customers_remaining',
              'customers_lost', 'first_order_moved', 'value_removed']]

,rule,what,rows_removed,rows_remaining,customers_remaining,customers_lost,first_order_moved,value_removed
0,Raw,file as received,0,1067371,5942,0,0,0.0
1,R1,No customer ID,243007,824364,5942,0,0,2638958.0
2,R2,Returns and cancellations,18744,805620,5881,61,214,-1095137.0
3,R3,Non-positive price or quantity,71,805549,5878,3,0,0.0
4,R4,Exact duplicate lines,26124,779425,5878,0,0,368625.0
5,R5,"Fee, service, adjustment and test lines",2148,777277,5868,10,10,154444.0


## Check records that were kept

**Checking unusual records**

Not every unusual transaction is necessarily incorrect. Before removing anything else, I checked a few groups of records that could potentially affect the results.

These checks help make sure that genuine purchases are not removed without a clear reason.

In [6]:
# Check Manual entries.
m = df[df['StockCode'] == 'M']

print('Manual lines kept:', len(m),'| customers:', m['Customer ID'].nunique(),'| value:', round(m['TotalPrice'].sum()))

# Check the less common product codes.
keep_odd = df[df['StockCode'].isin(['PADS', 'SP1002'])]
print('PADS / SP1002 lines kept:', len(keep_odd))

# Look at the customers with the highest total spending.
cust_spend = (df.groupby('Customer ID')['TotalPrice'].sum().sort_values(ascending=False))

print('Top 5 customers by spend:')
print(cust_spend.head(5).round(0).to_string())

print('Top 10 customers hold',round(cust_spend.head(10).sum() / cust_spend.sum() * 100, 1),'% of all spend')

Manual lines kept: 681 | customers: 438 | value: 151778
PADS / SP1002 lines kept: 19
Top 5 customers by spend:
Customer ID
18102    580987.0
14646    526752.0
14156    311248.0
14911    283632.0
17450    244784.0
Top 10 customers hold 16.1 % of all spend


In [7]:
# Returns have already been removed from df, but the original sales they cancelled are still in the file.
returns = raw[raw['Invoice'].str.startswith('C') & raw['Customer ID'].notna()].copy()
returns['value'] = returns['Quantity'] * returns['Price']                 # negative
ret_by_cust = returns.groupby(returns['Customer ID'].astype(int))['value'].sum()
gross = df.groupby('Customer ID')['TotalPrice'].sum()

# Compare gross spending with spending after returns.
net = (gross + ret_by_cust.reindex(gross.index).fillna(0))

print('return lines with a customer ID:', len(returns), '| value:', round(returns['value'].sum()))
print('gross spend (kept):', round(gross.sum()), '| net of returns:', round(net.sum()),
      '| gross overstates net by', round((gross.sum() / net.sum() - 1) * 100, 1), '%')
print('customers with any return:', int((ret_by_cust.reindex(gross.index).fillna(0) < 0).sum()), 'of', len(gross))
print('Spearman correlation, gross vs net spend per customer:', round(spearmanr(gross, net)[0], 3))

return lines with a customer ID: 18744 | value: -1095137
gross spend (kept): 17220360 | net of returns: 16186423 | gross overstates net by 6.4 %
customers with any return: 2510 of 5868
Spearman correlation, gross vs net spend per customer: 0.986


The return transactions are excluded from the cleaned dataset, but the original sales remain. As a result, the monetary values used later represent gross spending.

I also compared gross and net spending by customer using Spearman correlation. This gives an indication of whether including returns would change the ranking of customers by spending.

## Definitions written down
- **Order.** One customer on one day (`OrderDay`).
- **Snapshot dates.** Recency is measured back from the snapshot date. The scoring snapshot is the last timestamp in the data. The two backtest snapshots are 90 and 180 days earlier.
- **Observation window.** Days from a customer's first order to the snapshot. Frequency and spend grow with the window, so raw counts are not a fair comparison between customers with different windows.

In [8]:
max_date = df['InvoiceDate'].max()
snapshots = pd.DataFrame({'use': ['scoring (final)', 'backtest test', 'backtest train'],
                          'snapshot': [max_date, max_date - pd.Timedelta(days=90), max_date - pd.Timedelta(days=180)]})
print(snapshots.to_string(index=False))

# Customer observation windows
first = df.groupby('Customer ID')['InvoiceDate'].min()
window = (max_date - first).dt.days
print('\nobservation window (days) at the scoring snapshot: 25th', int(window.quantile(.25)),
      '| median', int(window.median()), '| 75th', int(window.quantile(.75)))
left_cens = int((first < '2010-01-01').sum())
print('customers whose first order in the file is in Dec 2009:', left_cens, f'({left_cens / len(first) * 100:.1f}%)  <- may not be new customers')

# Compare invoice counts with customer-day orders
orders_invoice = df.groupby('Customer ID')['Invoice'].nunique()
orders_day = df.groupby('Customer ID')['OrderDay'].nunique()
print('\norders as distinct invoices:', int(orders_invoice.sum()), '| as customer-days:', int(orders_day.sum()),
      '| customers whose count differs:', int((orders_invoice != orders_day).sum()))

            use            snapshot
scoring (final) 2011-12-09 12:50:00
  backtest test 2011-09-10 12:50:00
 backtest train 2011-06-12 12:50:00

observation window (days) at the scoring snapshot: 25th 311 | median 528 | 75th 666
customers whose first order in the file is in Dec 2009: 952 (16.2%)  <- may not be new customers

orders as distinct invoices: 36773 | as customer-days: 32980 | customers whose count differs: 1247


## Save the cleaned data and cleaning log

In [9]:
os.makedirs('data', exist_ok=True); os.makedirs('docs', exist_ok=True)
df.to_pickle('data/clean_transactions_v2.pkl')
print('saved', df.shape)

def md_table(frame):
    cols = list(frame.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    for _, r in frame.iterrows():
        lines.append('| ' + ' | '.join(f'{v:,.0f}' if isinstance(v, (int, float, np.integer, np.floating)) else str(v) for v in r) + ' |')
    return '\n'.join(lines)

show = cleaning_log[['rule', 'what', 'how_found', 'action', 'rows_removed', 'rows_remaining',
                     'customers_remaining', 'customers_lost', 'first_order_moved']]
final = cleaning_log.iloc[-1]
text = f'''# Cleaning log

Source: UCI Online Retail II dataset.
Original size: {len(raw):,} invoice lines.
Date range: {raw['InvoiceDate'].min():%Y-%m-%d} to {raw['InvoiceDate'].max():%Y-%m-%d}.

## Cleaning results

{md_table(show)}

The cleaned dataset contains {len(df):,} invoice lines
and {df['Customer ID'].nunique():,} customers.

## Records kept

- Manual (`M`) entries: {len(m):,} lines kept.
- `PADS` and `SP1002`: kept as product codes.
- High-spending customers: retained.

## Returns

Return lines were removed, but the original sales remain.
Gross spending is higher than net spending by
{(gross.sum() / net.sum() - 1) * 100:.1f}%.

The Spearman correlation between gross and net spending
per customer is {spearmanr(gross, net)[0]:.3f}.

## Definitions

- Order: one customer making purchases on one day.
- Scoring snapshot: {snapshots.iloc[0]['snapshot']:%Y-%m-%d}.
- Test snapshot: {snapshots.iloc[1]['snapshot']:%Y-%m-%d}.
- Training snapshot: {snapshots.iloc[2]['snapshot']:%Y-%m-%d}.

{left_cens:,} customers ({left_cens / len(first) * 100:.1f}%)
have their first recorded transaction in December 2009.
They may have purchased from the retailer before the dataset began.

## Limitations

- Transactions without a customer ID are excluded.
- Spending figures are gross rather than net of returns.
- The dataset covers one retailer from December 2009 to December 2011.
'''

with open('docs/cleaning-log.md', 'w') as f:
    f.write(text)

print(text[:1200])

saved (777277, 10)
# Cleaning log

Source: UCI Online Retail II dataset.
Original size: 1,067,371 invoice lines.
Date range: 2009-12-01 to 2011-12-09.

## Cleaning results

| rule | what | how_found | action | rows_removed | rows_remaining | customers_remaining | customers_lost | first_order_moved |
|---|---|---|---|---|---|---|---|---|
| Raw | file as received | - | - | 0 | 1,067,371 | 5,942 | 0 | 0 |
| R1 | No customer ID | Customer ID blank | removed | 243,007 | 824,364 | 5,942 | 0 | 0 |
| R2 | Returns and cancellations | Invoice starts with C | removed | 18,744 | 805,620 | 5,881 | 61 | 214 |
| R3 | Non-positive price or quantity | Price <= 0 or Quantity <= 0 | removed | 71 | 805,549 | 5,878 | 3 | 0 |
| R4 | Exact duplicate lines | all 8 columns identical to an earlier line | later copy removed | 26,124 | 779,425 | 5,878 | 0 | 0 |
| R5 | Fee, service, adjustment and test lines | StockCode in POST, C2, DOT, BANK CHARGES, ADJUST, ADJUST2, TEST001, TEST002, D | removed | 2,148 | 777,27